In [41]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

In [42]:
df = pd.read_csv('titanic_data_updated.csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,no,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,no,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,no,second,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,yes,first,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,no,third,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,yes,first,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


### splitting data

In [43]:
df.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)

# family_size creation
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

#feature and target extract
X = df.drop(['Survived'],axis=1)
y = df['Survived']


X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [44]:
# imputation transformer

imputer_transformer = ColumnTransformer(
    transformers = [
        ('age',SimpleImputer(missing_values=np.nan, strategy='mean'), ['Age']),
        ('embarkedd', SimpleImputer(missing_values=np.nan, strategy='most_frequent'), ['Embarked']),
        ('cabin', SimpleImputer(missing_values=np.nan, strategy='constant',fill_value='missing', add_indicator=True), ['Cabin']),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

imputer_transformer.set_output(transform='pandas')
imputer_transformer.fit(X_train)
X_train = imputer_transformer.transform(X_train)
X_train.isnull().sum()

Age                       0
Embarked                  0
Cabin                     0
missingindicator_Cabin    0
Pclass                    0
Sex                       0
SibSp                     0
Parch                     0
Fare                      0
FamilySize                0
dtype: int64

In [45]:
# outlier handling age

mean_of_age = X_train['Age'].mean()
std_of_age = X_train['Age'].std()
X_train['zscore'] = (X_train['Age'] - mean_of_age) / std_of_age

X_train = X_train[abs(X_train['zscore'] <= 3)]
X_train.drop(['zscore'], axis=1, inplace=True)

In [46]:
# outlier handling fare

fare_q1 = X_train['Fare'].quantile(0.25)
fare_q3 = X_train['Fare'].quantile(0.75)

fare_iqr = fare_q3 - fare_q1
fare_min = max(0, fare_q1 - 1.5 * fare_iqr)
fare_max = fare_q3 + 1.5 * fare_iqr

X_train['Fare'] = X_train['Fare'].clip(fare_min, fare_max)

In [47]:
# encoding and scaling

X_train['cabin_deck'] = X_train['Cabin'].astype(str).str[0]
X_test['cabin_deck'] = X_test['Cabin'].astype(str).str[0]

encoder_scaler = ColumnTransformer(
    transformers=[
        ('pclass', OrdinalEncoder(categories=[['third', 'second', 'first']]), ['Pclass']),
        ('embarked_sex_cabin', OneHotEncoder(sparse_output=False, drop='first').set_output(transform='pandas'), ['Embarked', 'Sex', 'cabin_deck']),
        ('age_scalling',StandardScaler(), ['Age']),
        ('fare_scaler', MinMaxScaler(), ['Fare']),
        ('familysize_scalling', MinMaxScaler(), ['FamilySize'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

encoder_scaler.set_output(transform='pandas')
encoder_scaler.fit(X_train)


X_train = encoder_scaler.transform(X_train)


In [48]:
X_train

,Pclass,Embarked_Q,Embarked_S,Sex_male,cabin_deck_B,cabin_deck_C,cabin_deck_D,cabin_deck_E,cabin_deck_F,cabin_deck_G,cabin_deck_T,cabin_deck_m,Age,Fare,FamilySize,Cabin,missingindicator_Cabin,SibSp,Parch
331,2.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.304495,0.442804,0.0,C124,False,0,0
733,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.495295,0.201981,0.0,missing,True,0,0
382,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.224621,0.123131,0.0,missing,True,0,0
704,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.255323,0.122031,0.1,missing,True,1,0
813,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-1.855135,0.485920,0.6,missing,True,4,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.655276,0.118858,0.0,missing,True,0,0
270,2.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.024552,0.481647,0.0,missing,True,0,0
860,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.944537,0.219201,0.2,missing,True,2,0
435,2.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.215210,1.000000,0.3,B96 B98,False,1,2
